In [1]:
!pip -q install sentence-transformers

In [2]:
import json
import numpy as np
import pandas as pd

from sentence_transformers import SentenceTransformer
from sklearn.metrics.pairwise import cosine_similarity

In [3]:
candidate_scores = pd.read_csv("candidate_scores.csv")
candidate_features = pd.read_csv("candidate_features.csv")

with open("jd_profile.json","r") as f:
    jd = json.load(f)

In [4]:
model = SentenceTransformer(
    "all-MiniLM-L6-v2"
)

modules.json:   0%|          | 0.00/349 [00:00<?, ?B/s]

config_sentence_transformers.json:   0%|          | 0.00/116 [00:00<?, ?B/s]

README.md:   0%|          | 0.00/10.5k [00:00<?, ?B/s]

sentence_bert_config.json:   0%|          | 0.00/53.0 [00:00<?, ?B/s]

config.json:   0%|          | 0.00/612 [00:00<?, ?B/s]

...

tokenizer_config.json:   0%|          | 0.00/350 [00:00<?, ?B/s]

vocab.txt:   0%|          | 0.00/232k [00:00<?, ?B/s]

tokenizer.json:   0%|          | 0.00/466k [00:00<?, ?B/s]

special_tokens_map.json:   0%|          | 0.00/112 [00:00<?, ?B/s]

config.json:   0%|          | 0.00/190 [00:00<?, ?B/s]

In [5]:
jd_text = " ".join(jd["skills"])

jd_text += " "

jd_text += " ".join(jd["production"])

jd_text += " "

jd_text += " ".join(jd["mandatory"])

In [6]:
candidate_text = []

for _, row in candidate_features.iterrows():

    text = f"""
    Experience {row['experience']}
    Production {row['production_score']}
    Skills {row['production_ai_skills']}
    Career {row['career_length']}
    """

    candidate_text.append(text)

In [7]:
jd_embedding = model.encode(
    [jd_text]
)

candidate_embeddings = model.encode(
    candidate_text,
    batch_size=64,
    show_progress_bar=True
)

Batches:   0%|          | 0/28 [00:00<?, ?it/s]

In [8]:
semantic_scores = cosine_similarity(

    candidate_embeddings,

    jd_embedding

).flatten()

candidate_scores["semantic_score"] = semantic_scores

In [9]:
candidate_scores["semantic_score"] = (

candidate_scores["semantic_score"]

-

candidate_scores["semantic_score"].min()

)/(

candidate_scores["semantic_score"].max()

-

candidate_scores["semantic_score"].min()

)

In [10]:
candidate_scores["ranking_score"] = (

0.60 * candidate_scores["semantic_score"]

+

0.40 * candidate_scores["final_score"]

)

In [11]:
candidate_scores = candidate_scores.sort_values(

"ranking_score",

ascending=False

).reset_index(drop=True)

candidate_scores["rank"] = np.arange(

1,

len(candidate_scores)+1

)

In [12]:
candidate_scores.head(20)

,candidate_id,experience,num_skills,career_length,education_count,avg_skill_duration,total_endorsements,github,profile_complete,response_rate,...,behavior_score,career_score,innovation_score,penalty,recruiter_score,confidence,final_score,rank,semantic_score,ranking_score
0,CAND_0001085,0.321429,14,0.375,2,21.714286,167,0.869403,0.538356,0.206897,...,0.389098,0.299256,1.0,0.05,0.486390,0.511438,0.490148,1,0.915436,0.745321
1,CAND_0001182,0.585714,6,0.250,1,16.833333,35,0.000000,0.760274,0.011494,...,0.408969,0.477579,0.4,0.00,0.290332,0.760274,0.360823,2,0.970471,0.726612
2,CAND_0001069,0.635714,9,0.250,1,21.000000,93,0.000000,0.441096,0.816092,...,0.470752,0.513690,0.4,0.00,0.294573,0.441096,0.316551,3,0.967906,0.707364
3,CAND_0001109,0.664286,17,0.500,1,13.470588,91,0.329602,0.646575,0.103448,...,0.345317,0.540476,0.4,0.00,0.384306,0.646575,0.423647,4,0.890387,0.703691
4,CAND_0000859,0.350000,12,0.125,1,26.083333,165,0.936567,0.863014,0.839080,...,0.608644,0.316667,0.4,0.00,0.419569,0.863014,0.486086,5,0.832874,0.694159
5,CAND_0001482,0.321429,12,0.125,2,14.416667,109,0.000000,0.853425,0.114943,...,0.513957,0.291964,0.4,0.00,0.346068,0.853425,0.422172,6,0.855062,0.681906
6,CAND_0000874,0.378571,13,0.250,1,18.307692,169,0.000000,0.515068,0.632184,...,0.491258,0.326786,1.0,0.00,0.457091,0.515068,0.465788,7,0.821865,0.679434
7,CAND_0001598,0.678571,5,0.500,2,11.800000,34,0.000000,0.719178,0.103448,...,0.383882,0.550119,0.4,0.00,0.303449,0.719178,0.365808,8,0.885380,0.677551
8,CAND_0001153,0.721429,8,0.375,1,18.375000,77,0.000000,0.284932,0.678161,...,0.320587,0.568006,0.4,0.00,0.278119,0.284932,0.279141,9,0.941188,0.676369
9,CAND_0000510,0.642857,14,0.375,2,19.000000,94,0.000000,0.353425,0.793103,...,0.374212,0.516220,0.2,0.00,0.354183,0.353425,0.354069,10,0.885252,0.672779


In [13]:
candidate_scores.to_csv(

"ranked_candidates.csv",

index=False

)

print(candidate_scores.shape)

(1734, 35)
